# BioFuse Tutorial 5: Adding a New Fusion Method

This tutorial shows you how to add custom fusion methods to combine embeddings from multiple foundation models.

We'll cover:
1. Understanding BioFuse's fusion architecture
2. Existing fusion methods
3. Creating simple fusion methods
4. Creating advanced fusion methods with learnable parameters
5. Testing and benchmarking fusion methods

## What is Fusion?

When BioFuse uses multiple foundation models (e.g., BioMedCLIP + CONCH), each model produces an embedding vector. **Fusion** is the process of combining these multiple embeddings into a single unified representation for downstream classification.

Example:
- BioMedCLIP produces 512-dim embedding
- CONCH produces 512-dim embedding
- **Fusion** combines them → single embedding for classifier

## Part 1: Understanding the Fusion Architecture

Fusion happens in `biofuse/models/biofuse_model.py` in the `BioFuseModel` class.

### Key Components

1. **Projection layers** (optional): Project each model's embeddings to a common dimension
2. **Fusion method**: Combine the (projected) embeddings
3. **Forward pass**: Apply projection → fusion → return unified embedding

### Current Fusion Methods

BioFuse supports 9 built-in fusion methods:

In [ ]:
# Built-in fusion methods:

fusion_methods = {
    # Simple methods (no learnable parameters)
    'concat': 'Concatenate embeddings [emb1 | emb2]',
    'mean': 'Average embeddings element-wise',
    'max': 'Element-wise maximum',
    'sum': 'Element-wise sum',
    'mul': 'Element-wise multiplication',
    
    # Weighted methods (learnable weights)
    'wsum': 'Weighted sum with learnable weights',
    'wmean': 'Weighted mean with learnable weights',
    
    # Advanced methods
    'ifusion': 'Interleaved fusion (chunk-based)',
    'self_attention': 'Multi-head self-attention fusion'
}

for method, description in fusion_methods.items():
    print(f"{method:15s} - {description}")

## Part 2: Adding a Simple Fusion Method

Let's add a new fusion method called **"harmonic_mean"** that computes the harmonic mean of embeddings.

### Step 1: Locate the fusion code

Open `biofuse/models/biofuse_model.py` and find the `forward()` method (around line 83).

In [ ]:
# Current forward() method in biofuse_model.py:

"""
def forward(self, embeddings):
    # Project embeddings
    embeddings = [projection(embedding) for embedding, projection 
                  in zip(embeddings, self.projection_layers)]
    
    # Fusion logic
    if self.fusion_method == 'concat':
        fused_embedding = torch.cat(embeddings, dim=-1)
    elif self.fusion_method == 'mean':
        fused_embedding = torch.mean(torch.stack(embeddings), dim=0)
    # ... other methods ...
    else:
        raise ValueError(f'Fusion method {self.fusion_method} not supported')
    
    return fused_embedding
"""

print("This is the structure we'll modify")

### Step 2: Add harmonic mean fusion

Add this code to the `forward()` method in `biofuse/models/biofuse_model.py`:

In [ ]:
# Add to forward() method after other fusion methods:

"""
elif self.fusion_method == 'harmonic_mean':
    # Harmonic mean: n / (1/x1 + 1/x2 + ... + 1/xn)
    # Add small epsilon to avoid division by zero
    epsilon = 1e-8
    stacked_embeddings = torch.stack(embeddings)
    reciprocal_sum = torch.sum(1.0 / (stacked_embeddings + epsilon), dim=0)
    fused_embedding = len(embeddings) / (reciprocal_sum + epsilon)
"""

print("Add this to the forward() method")

### Step 3: Also update forward_test()

The same logic should be added to `forward_test()` method (around line 125):

In [ ]:
# Add to forward_test() method:

"""
elif self.fusion_method == 'harmonic_mean':
    epsilon = 1e-8
    stacked_embeddings = torch.stack(projected_embeddings)
    reciprocal_sum = torch.sum(1.0 / (stacked_embeddings + epsilon), dim=0)
    fused_embedding = len(projected_embeddings) / (reciprocal_sum + epsilon)
"""

print("Also add this to forward_test()")

### Step 4: Test the new fusion method

In [ ]:
from biofuse import BioFuse, load_medmnist, get_classifier, compute_metrics

# Load dataset
train_data, num_classes = load_medmnist('pathmnist', split='train')
test_data, _ = load_medmnist('pathmnist', split='test')

# Use NEW fusion method
biofuse = BioFuse(
    models=['BioMedCLIP', 'CONCH'],
    fusion_method='harmonic_mean'  # <-- Our new method!
)

# Extract embeddings
train_emb, train_labels, _, _, _ = biofuse.generate_embeddings(
    train_data=None,
    dataset_type='medmnist',
    dataset_name='pathmnist',
    split='train'
)

test_emb, test_labels, _, _, _ = biofuse.generate_embeddings(
    train_data=None,
    dataset_type='medmnist',
    dataset_name='pathmnist',
    split='test'
)

print(f"Train embeddings shape: {train_emb.shape}")
print(f"Test embeddings shape: {test_emb.shape}")

# Train classifier
clf = get_classifier('xgboost', num_classes=num_classes)
clf.fit(train_emb, train_labels)

# Evaluate
pred = clf.predict(test_emb)
proba = clf.predict_proba(test_emb)
metrics = compute_metrics(test_labels, pred, proba, num_classes, task='multi-class')

print(f"\nHarmonic Mean Fusion - Accuracy: {metrics['accuracy']:.4f}")

## Part 3: Advanced Fusion with Learnable Parameters

Let's create a more sophisticated fusion method with learnable parameters: **"gated_fusion"**.

This method learns gates to dynamically weight different models per sample.

### Step 1: Initialize learnable parameters in `__init__`

Add to the `__init__` method in `BioFuseModel` class (around line 37):

In [ ]:
# Add to __init__ method after other fusion initializations:

"""
elif self.fusion_method == 'gated_fusion':
    # Gating network: learns to weight models dynamically
    # For each model, learn a gate that depends on the embedding
    self.gate_networks = nn.ModuleList([
        nn.Sequential(
            nn.Linear(self.projection_dim if self.projection_dim > 0 
                     else self.get_model_dim(model), 1),
            nn.Sigmoid()
        )
        for model in [m.model_name for m in models]
    ])
"""

print("Add this to __init__()")

### Step 2: Implement gated fusion in forward()

Add to the `forward()` method:

In [ ]:
# Add to forward() method:

"""
elif self.fusion_method == 'gated_fusion':
    # Compute gates for each embedding
    gates = [gate_net(emb) for gate_net, emb in zip(self.gate_networks, embeddings)]
    
    # Apply gates (element-wise multiplication)
    gated_embeddings = [gate * emb for gate, emb in zip(gates, embeddings)]
    
    # Sum gated embeddings
    fused_embedding = torch.sum(torch.stack(gated_embeddings), dim=0)
"""

print("Add this to forward()")

### Step 3: Also add to forward_test()

In [ ]:
# Add to forward_test():

"""
elif self.fusion_method == 'gated_fusion':
    gates = [gate_net(emb) for gate_net, emb in zip(self.gate_networks, projected_embeddings)]
    gated_embeddings = [gate * emb for gate, emb in zip(gates, projected_embeddings)]
    fused_embedding = torch.sum(torch.stack(gated_embeddings), dim=0)
"""

print("Add this to forward_test()")

## Part 4: Example Fusion Methods

Here are more fusion method ideas you can implement:

### A) Geometric Mean

In [ ]:
# Geometric mean fusion
"""
elif self.fusion_method == 'geometric_mean':
    # Geometric mean: (x1 * x2 * ... * xn)^(1/n)
    stacked_embeddings = torch.stack(embeddings)
    # Use log for numerical stability: exp(mean(log(x)))
    epsilon = 1e-8
    log_embeddings = torch.log(torch.abs(stacked_embeddings) + epsilon)
    fused_embedding = torch.exp(torch.mean(log_embeddings, dim=0))
"""

print("Geometric mean fusion")

### B) Attention-based Fusion (Cross-attention)

In [ ]:
# In __init__:
"""
elif self.fusion_method == 'cross_attention':
    self.query_projection = nn.Linear(self.projection_dim, self.projection_dim)
    self.key_projection = nn.Linear(self.projection_dim, self.projection_dim)
    self.value_projection = nn.Linear(self.projection_dim, self.projection_dim)
"""

# In forward():
"""
elif self.fusion_method == 'cross_attention':
    # Stack embeddings [batch, num_models, dim]
    stacked = torch.stack(embeddings, dim=1)
    
    # Compute attention
    Q = self.query_projection(stacked)
    K = self.key_projection(stacked)
    V = self.value_projection(stacked)
    
    attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.projection_dim ** 0.5)
    attention_weights = F.softmax(attention_scores, dim=-1)
    
    attended = torch.matmul(attention_weights, V)
    fused_embedding = attended.mean(dim=1)
"""

print("Cross-attention fusion")

### C) Tucker Decomposition Fusion

In [ ]:
# In __init__:
"""
elif self.fusion_method == 'tucker':
    # Tucker decomposition for compact fusion
    self.core_tensor = nn.Parameter(
        torch.randn(len(models), self.projection_dim, self.projection_dim)
    )
"""

# In forward():
"""
elif self.fusion_method == 'tucker':
    stacked = torch.stack(embeddings, dim=0)  # [num_models, batch, dim]
    # Contract with core tensor
    fused_embedding = torch.einsum('mbd,mdk->bk', stacked, self.core_tensor)
"""

print("Tucker decomposition fusion")

## Part 5: Benchmarking Fusion Methods

Let's compare different fusion methods on the same task:

In [ ]:
import matplotlib.pyplot as plt
from biofuse import BioFuse, load_medmnist, get_classifier, compute_metrics

# Load data once
train_data, num_classes = load_medmnist('pathmnist', split='train')
test_data, _ = load_medmnist('pathmnist', split='test')

# Test different fusion methods
fusion_methods_to_test = ['concat', 'mean', 'max', 'sum', 'wsum', 'self_attention']
results = {}

for fusion_method in fusion_methods_to_test:
    print(f"\nTesting {fusion_method}...")
    
    # Initialize BioFuse with fusion method
    biofuse = BioFuse(
        models=['BioMedCLIP', 'CONCH'],
        fusion_method=fusion_method,
        projection_dim=256 if fusion_method == 'self_attention' else 0
    )
    
    # Extract embeddings
    train_emb, train_labels, _, _, _ = biofuse.generate_embeddings(
        train_data=None,
        dataset_type='medmnist',
        dataset_name='pathmnist',
        split='train'
    )
    
    test_emb, test_labels, _, _, _ = biofuse.generate_embeddings(
        train_data=None,
        dataset_type='medmnist',
        dataset_name='pathmnist',
        split='test'
    )
    
    # Train classifier
    clf = get_classifier('xgboost', num_classes=num_classes)
    clf.fit(train_emb, train_labels)
    
    # Evaluate
    pred = clf.predict(test_emb)
    proba = clf.predict_proba(test_emb)
    metrics = compute_metrics(test_labels, pred, proba, num_classes, task='multi-class')
    
    results[fusion_method] = metrics['accuracy']
    print(f"  Accuracy: {metrics['accuracy']:.4f}")

# Plot comparison
plt.figure(figsize=(12, 6))
plt.bar(results.keys(), results.values())
plt.ylabel('Accuracy')
plt.xlabel('Fusion Method')
plt.title('Fusion Method Comparison on PathMNIST')
plt.ylim([0.8, 1.0])
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (method, acc) in enumerate(results.items()):
    plt.text(i, acc + 0.01, f'{acc:.4f}', ha='center')

plt.tight_layout()
plt.show()

# Print summary
print("\n" + "="*50)
print("FUSION METHOD BENCHMARK SUMMARY")
print("="*50)
sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
for i, (method, acc) in enumerate(sorted_results, 1):
    print(f"{i}. {method:15s} - {acc:.4f}")
print("="*50)

## Part 6: Complete Example - Adding "Pooled Attention" Fusion

Let's implement a complete fusion method from scratch: **pooled attention fusion**.

This method:
1. Learns attention weights for each model
2. Applies pooling before attention
3. Returns weighted combination

### Full implementation:

In [ ]:
# ============================================
# ADD TO __init__ in BioFuseModel:
# ============================================
"""
elif self.fusion_method == 'pooled_attention':
    # Attention pooling network
    dim = self.projection_dim if self.projection_dim > 0 else 512
    self.attention_pooling = nn.Sequential(
        nn.Linear(dim, dim // 2),
        nn.Tanh(),
        nn.Linear(dim // 2, 1)
    )
"""

# ============================================
# ADD TO forward() in BioFuseModel:
# ============================================
"""
elif self.fusion_method == 'pooled_attention':
    # Stack embeddings [num_models, batch, dim]
    stacked = torch.stack(embeddings, dim=0)
    
    # Compute attention scores for each model
    # scores shape: [num_models, batch, 1]
    scores = torch.stack([self.attention_pooling(emb) for emb in embeddings])
    
    # Softmax over models dimension
    weights = F.softmax(scores, dim=0)
    
    # Weighted sum
    fused_embedding = torch.sum(stacked * weights, dim=0)
"""

# ============================================
# ADD TO forward_test() in BioFuseModel:
# ============================================
"""
elif self.fusion_method == 'pooled_attention':
    stacked = torch.stack(projected_embeddings, dim=0)
    scores = torch.stack([self.attention_pooling(emb) for emb in projected_embeddings])
    weights = F.softmax(scores, dim=0)
    fused_embedding = torch.sum(stacked * weights, dim=0)
"""

print("Complete implementation for pooled_attention fusion")

## Summary

You now know how to add custom fusion methods to BioFuse!

### Checklist for Adding a Fusion Method

- [ ] Decide on fusion strategy (simple/learnable)
- [ ] Add initialization code to `__init__()` (if learnable parameters)
- [ ] Implement fusion logic in `forward()`
- [ ] Implement same logic in `forward_test()`
- [ ] Test on small dataset
- [ ] Benchmark against existing methods
- [ ] Document expected input/output shapes

### Key Considerations

1. **Shape compatibility**: All fusion methods receive a list of embeddings, each of shape `[batch_size, embedding_dim]`
2. **Projection**: Embeddings are already projected before fusion (if `projection_dim > 0`)
3. **Learnable parameters**: Initialize in `__init__`, use in `forward()`
4. **Numerical stability**: Add epsilon for divisions, use log for products
5. **Memory efficiency**: Use in-place operations when possible

### Fusion Method Design Tips

**Simple methods** (no parameters):
- Fast inference
- No training overhead
- Good baseline
- Examples: concat, mean, max

**Learnable methods** (with parameters):
- Can adapt to data
- Requires training
- More expressive
- Examples: wsum, self_attention, gated

## Next Steps

Try implementing these fusion methods:
1. **Min fusion**: Element-wise minimum
2. **Stacked MLP**: Small MLP to combine embeddings
3. **Bilinear fusion**: Outer product + pooling
4. **Transformer fusion**: Full transformer layer

## Resources

- [BioFuse Model Code](../../biofuse/models/biofuse_model.py)
- [Multimodal Fusion Survey](https://arxiv.org/abs/2007.09200)
- [Attention Mechanisms](https://arxiv.org/abs/1706.03762)